<a href="https://colab.research.google.com/github/kabir-codes/ml-challenge/blob/main/ML_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MultiLabelBinarizer
import re

In [ ]:
df = pd.read_csv("/content/ml_challenge_dataset.csv")
df

,unique_id,Painting,"On a scale of 1–10, how intense is the emotion conveyed by the artwork?",Describe how this painting makes you feel.,This art piece makes me feel sombre.,This art piece makes me feel content.,This art piece makes me feel calm.,This art piece makes me feel uneasy.,How many prominent colours do you notice in this painting?,How many objects caught your eye in the painting?,How much (in Canadian dollars) would you be willing to pay for this painting?,"If you could purchase this painting, which room would you put that painting in?","If you could view this art in person, who would you want to view it with?",What season does this art piece remind you of?,"If this painting was a food, what would be?",Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.
0,1,The Persistence of Memory,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,The Persistence of Memory,5.0,"The clocks are burnt on a hot desert, it embod...",4 - Agree,3 - Neutral/Unsure,2 - Disagree,1 - Strongly disagree,2.0,4.0,0,Bathroom,By yourself,Fall,Fries,A country song that contrasts nostalgia for th...
2,3,The Persistence of Memory,7.0,This painting makes me feel dread. The clock r...,4 - Agree,1 - Strongly disagree,1 - Strongly disagree,4 - Agree,4.0,3.0,$5,"Bathroom,Dining room","Coworkers/Classmates,By yourself",Fall,Sardines,A melancholy instrumental with a monotone voic...
3,4,The Persistence of Memory,7.0,Deflated,4 - Agree,1 - Strongly disagree,2 - Disagree,4 - Agree,10.0,7.0,a,"Bedroom,Bathroom",Coworkers/Classmates,Winter,a,q
4,5,The Persistence of Memory,7.0,The painting gives me a sense of calmness and ...,3 - Neutral/Unsure,4 - Agree,5 - Strongly agree,3 - Neutral/Unsure,4.0,6.0,300 dollars.,Living room,Friends,"Spring,Summer",Churros.,Radiohead's album in rainbows.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,558,The Water Lily Pond,5.0,Joy and relaxation.,2 - Disagree,4 - Agree,4 - Agree,2 - Disagree,2.0,1.0,$35,"Bedroom,Office,Living room,Dining room","Friends,Family members,Coworkers/Classmates,St...",Spring,green apple,nature forest soundtrack
1682,559,The Water Lily Pond,9.0,This painting makes me feel happy,2 - Disagree,5 - Strongly agree,5 - Strongly agree,1 - Strongly disagree,3.0,3.0,200,"Bedroom,Office,Living room",Friends,"Spring,Summer",Salad,Slow guitar soundtrack
1683,560,The Water Lily Pond,8.0,This painting makes me feel warm and new birth...,1 - Strongly disagree,5 - Strongly agree,3 - Neutral/Unsure,1 - Strongly disagree,2.0,2.0,"200 Dollars, pay for the emotion","Bedroom,Living room","Friends,Family members",Spring,Fresh salad,"Various animal sounds, like bird sounds; and l..."
1684,561,The Water Lily Pond,5.0,makes me feel nonchalant,2 - Disagree,2 - Disagree,4 - Agree,1 - Strongly disagree,4.0,1.0,10,"Bathroom,Office","Friends,Family members,Coworkers/Classmates,St...",Spring,apple,simon and garfunkel


In [ ]:
# Renaming Column names

df = df.rename(columns={'unique_id': 'rid',
                        'Painting': 'label', 'On a scale of 1–10, how intense is the emotion conveyed by the artwork?': 'intensity',
                        'Describe how this painting makes you feel.': 'feeling',
                        'This art piece makes me feel sombre.': 'sombre?',
                        'This art piece makes me feel content.': 'content?',
                        'This art piece makes me feel calm.': 'calm?',
                        'This art piece makes me feel uneasy.': 'uneasy?',
                        'How many prominent colours do you notice in this painting?': 'num_colors',
                        'How many objects caught your eye in the painting?': 'num_objects',
                        'How much (in Canadian dollars) would you be willing to pay for this painting?': 'worth',
                        'If you could purchase this painting, which room would you put that painting in?': 'place',
                        'If you could view this art in person, who would you want to view it with?': 'view_with',
                        'What season does this art piece remind you of?': 'season',
                        'If this painting was a food, what would be?': 'food',
                        'Imagine a soundtrack for this painting. Describe that soundtrack without naming any objects in the painting.': 'music'})

In [ ]:
# Aarushi
#counting number of na in the data and total number of rows in intensity
print(df['intensity'].isna().sum())
print(len(df['intensity']))
# replacing N/A in intensity column with mean/median of the column.
df['intensity'] = df['intensity'].fillna(df['intensity'].median())
df

65
1686


,rid,label,intensity,feeling,sombre?,content?,calm?,uneasy?,num_colors,num_objects,worth,place,view_with,season,food,music
0,1,The Persistence of Memory,7.0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,The Persistence of Memory,5.0,"The clocks are burnt on a hot desert, it embod...",4 - Agree,3 - Neutral/Unsure,2 - Disagree,1 - Strongly disagree,2.0,4.0,0,Bathroom,By yourself,Fall,Fries,A country song that contrasts nostalgia for th...
2,3,The Persistence of Memory,7.0,This painting makes me feel dread. The clock r...,4 - Agree,1 - Strongly disagree,1 - Strongly disagree,4 - Agree,4.0,3.0,$5,"Bathroom,Dining room","Coworkers/Classmates,By yourself",Fall,Sardines,A melancholy instrumental with a monotone voic...
3,4,The Persistence of Memory,7.0,Deflated,4 - Agree,1 - Strongly disagree,2 - Disagree,4 - Agree,10.0,7.0,a,"Bedroom,Bathroom",Coworkers/Classmates,Winter,a,q
4,5,The Persistence of Memory,7.0,The painting gives me a sense of calmness and ...,3 - Neutral/Unsure,4 - Agree,5 - Strongly agree,3 - Neutral/Unsure,4.0,6.0,300 dollars.,Living room,Friends,"Spring,Summer",Churros.,Radiohead's album in rainbows.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,558,The Water Lily Pond,5.0,Joy and relaxation.,2 - Disagree,4 - Agree,4 - Agree,2 - Disagree,2.0,1.0,$35,"Bedroom,Office,Living room,Dining room","Friends,Family members,Coworkers/Classmates,St...",Spring,green apple,nature forest soundtrack
1682,559,The Water Lily Pond,9.0,This painting makes me feel happy,2 - Disagree,5 - Strongly agree,5 - Strongly agree,1 - Strongly disagree,3.0,3.0,200,"Bedroom,Office,Living room",Friends,"Spring,Summer",Salad,Slow guitar soundtrack
1683,560,The Water Lily Pond,8.0,This painting makes me feel warm and new birth...,1 - Strongly disagree,5 - Strongly agree,3 - Neutral/Unsure,1 - Strongly disagree,2.0,2.0,"200 Dollars, pay for the emotion","Bedroom,Living room","Friends,Family members",Spring,Fresh salad,"Various animal sounds, like bird sounds; and l..."
1684,561,The Water Lily Pond,5.0,makes me feel nonchalant,2 - Disagree,2 - Disagree,4 - Agree,1 - Strongly disagree,4.0,1.0,10,"Bathroom,Office","Friends,Family members,Coworkers/Classmates,St...",Spring,apple,simon and garfunkel


In [ ]:
# Adding column class

loc_index = df.columns.get_loc('label') + 1
new_col_values = []
for i in range(len(df)):
  j = i // 562
  new_col_values.append('{}'.format(j))
df.insert(loc=loc_index, column='Class', value=new_col_values)
df

,rid,label,Class,intensity,feeling,sombre?,content?,calm?,uneasy?,num_colors,num_objects,worth,place,view_with,season,food,music
0,1,The Persistence of Memory,0,7.0,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,The Persistence of Memory,0,5.0,"The clocks are burnt on a hot desert, it embod...",4 - Agree,3 - Neutral/Unsure,2 - Disagree,1 - Strongly disagree,2.0,4.0,0,Bathroom,By yourself,Fall,Fries,A country song that contrasts nostalgia for th...
2,3,The Persistence of Memory,0,7.0,This painting makes me feel dread. The clock r...,4 - Agree,1 - Strongly disagree,1 - Strongly disagree,4 - Agree,4.0,3.0,$5,"Bathroom,Dining room","Coworkers/Classmates,By yourself",Fall,Sardines,A melancholy instrumental with a monotone voic...
3,4,The Persistence of Memory,0,7.0,Deflated,4 - Agree,1 - Strongly disagree,2 - Disagree,4 - Agree,10.0,7.0,a,"Bedroom,Bathroom",Coworkers/Classmates,Winter,a,q
4,5,The Persistence of Memory,0,7.0,The painting gives me a sense of calmness and ...,3 - Neutral/Unsure,4 - Agree,5 - Strongly agree,3 - Neutral/Unsure,4.0,6.0,300 dollars.,Living room,Friends,"Spring,Summer",Churros.,Radiohead's album in rainbows.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,558,The Water Lily Pond,2,5.0,Joy and relaxation.,2 - Disagree,4 - Agree,4 - Agree,2 - Disagree,2.0,1.0,$35,"Bedroom,Office,Living room,Dining room","Friends,Family members,Coworkers/Classmates,St...",Spring,green apple,nature forest soundtrack
1682,559,The Water Lily Pond,2,9.0,This painting makes me feel happy,2 - Disagree,5 - Strongly agree,5 - Strongly agree,1 - Strongly disagree,3.0,3.0,200,"Bedroom,Office,Living room",Friends,"Spring,Summer",Salad,Slow guitar soundtrack
1683,560,The Water Lily Pond,2,8.0,This painting makes me feel warm and new birth...,1 - Strongly disagree,5 - Strongly agree,3 - Neutral/Unsure,1 - Strongly disagree,2.0,2.0,"200 Dollars, pay for the emotion","Bedroom,Living room","Friends,Family members",Spring,Fresh salad,"Various animal sounds, like bird sounds; and l..."
1684,561,The Water Lily Pond,2,5.0,makes me feel nonchalant,2 - Disagree,2 - Disagree,4 - Agree,1 - Strongly disagree,4.0,1.0,10,"Bathroom,Office","Friends,Family members,Coworkers/Classmates,St...",Spring,apple,simon and garfunkel


In [ ]:
# One-hot encoding for place, view_with, season column

# One-hot encoding for place column
df['place_list'] = df['place'].fillna('').str.split(',')

mlb = MultiLabelBinarizer()

encoded_place = pd.DataFrame(
    mlb.fit_transform(df['place_list']),
    columns=mlb.classes_,
    index=df.index
)

df = pd.concat([df, encoded_place], axis=1)
df.drop(columns=['place', 'place_list'], inplace=True)

# One-hot encoding for view_with column
df['view_with_list'] = df['view_with'].fillna('').str.split(',')
encoded_view_with = pd.DataFrame(
    mlb.fit_transform(df['view_with_list']),
    columns=mlb.classes_,
    index=df.index
)
df = pd.concat([df, encoded_view_with], axis=1)

# One-hot encoding for season column
df['season_list'] = df['season'].fillna('').str.split(',')
encoded_season = pd.DataFrame(
    mlb.fit_transform(df['season_list']),
    columns=mlb.classes_,
    index=df.index
)
df = pd.concat([df, encoded_season], axis=1)
df.drop(columns=['view_with', 'view_with_list', 'season', 'season_list'], inplace=True)

df.head()

,rid,label,Class,intensity,feeling,sombre?,content?,calm?,uneasy?,num_colors,...,By yourself,Coworkers/Classmates,Family members,Friends,Strangers,,Fall,Spring,Summer,Winter
0,1,The Persistence of Memory,0,7.0,NaN,NaN,NaN,NaN,NaN,5.0,...,0,0,0,0,0,1,0,0,0,0
1,2,The Persistence of Memory,0,5.0,"The clocks are burnt on a hot desert, it embod...",4 - Agree,3 - Neutral/Unsure,2 - Disagree,1 - Strongly disagree,2.0,...,1,0,0,0,0,0,1,0,0,0
2,3,The Persistence of Memory,0,7.0,This painting makes me feel dread. The clock r...,4 - Agree,1 - Strongly disagree,1 - Strongly disagree,4 - Agree,4.0,...,1,1,0,0,0,0,1,0,0,0
3,4,The Persistence of Memory,0,7.0,Deflated,4 - Agree,1 - Strongly disagree,2 - Disagree,4 - Agree,10.0,...,0,1,0,0,0,0,0,0,0,1
4,5,The Persistence of Memory,0,7.0,The painting gives me a sense of calmness and ...,3 - Neutral/Unsure,4 - Agree,5 - Strongly agree,3 - Neutral/Unsure,4.0,...,0,0,0,1,0,0,0,1,1,0


In [ ]:
# Aarushi

# replacing N/A in numerical columns with mean/median of the column.
df['intensity'] = df['intensity'].fillna(df['intensity'].median())
df['num_colors'] = df['num_colors'].fillna(df['num_colors'].median())
df['num_objects'] = df['num_objects'].fillna(df['num_objects'].median())

cols = ['sombre?', 'content?', 'calm?', 'uneasy?']
for col in cols:
    df[col] = df[col].str.extract(r'(\d+)').astype(float)

for col in cols:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
# cleaning the feeling column and applying TF-IDF vectorization to it.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stops = ['feel', 'feeling', 'feels', 'gives', 'makes',
                'make', 'like', 'bit', 'sense', 'world',
                'painting', 'little', 'life', 'away', 'look',
                'looking', 'really', 'reminds', 'think', 'way', 'wonder', 'want']

vectorizer = TfidfVectorizer(
    max_features=50,
    stop_words=list(ENGLISH_STOP_WORDS) + custom_stops  # combine with built-in stops
)

feeling_tfidf = vectorizer.fit_transform(df['feeling'].fillna(''))

feeling_df = pd.DataFrame(feeling_tfidf.toarray(),
                          columns=vectorizer.get_feature_names_out())
df = pd.concat([df.reset_index(drop=True), feeling_df], axis=1)
df = df.drop(columns=['feeling']) # drop the old feelings column

In [ ]:
print(df.columns)
print(vectorizer.get_feature_names_out())
feeling_cols = [col for col in df.columns if col in vectorizer.get_feature_names_out()]
print(df[feeling_cols].iloc[1])
len(df.columns)
df = df.loc[:, ~df.columns.duplicated(keep=False)]

Index(['rid', 'label', 'Class', 'intensity', 'sombre?', 'content?', 'calm?',
       'uneasy?', 'num_colors', 'num_objects', 'worth', 'food', 'music', '',
       'Bathroom', 'Bedroom', 'Dining room', 'Living room', 'Office', '',
       'By yourself', 'Coworkers/Classmates', 'Family members', 'Friends',
       'Strangers', '', 'Fall', 'Spring', 'Summer', 'Winter', 'anxious', 'awe',
       'beautiful', 'beauty', 'bright', 'calm', 'clock', 'clocks', 'cold',
       'colors', 'colours', 'confused', 'content', 'dark', 'day', 'excited',
       'good', 'happy', 'hopeful', 'joyful', 'just', 'light', 'lonely',
       'looks', 'melancholic', 'melting', 'memories', 'moving', 'nature',
       'night', 'nostalgic', 'passing', 'peace', 'peaceful', 'quiet',
       'relaxed', 'running', 'sad', 'scene', 'serene', 'sky', 'slightly',
       'strange', 'summer', 'things', 'time', 'uncomfortable', 'uneasy',
       'warm', 'weird'],
      dtype='object')
['anxious' 'awe' 'beautiful' 'beauty' 'bright' 'calm' '

In [ ]:
df[['happy', 'label']].iloc[100:120]

,happy,label
100,0.0,The Persistence of Memory
101,0.0,The Persistence of Memory
102,0.0,The Persistence of Memory
103,0.0,The Persistence of Memory
104,0.0,The Persistence of Memory
105,0.0,The Persistence of Memory
106,0.0,The Persistence of Memory
107,0.0,The Persistence of Memory
108,0.0,The Persistence of Memory
109,0.0,The Persistence of Memory


In [ ]:
feeling_cols = [col for col in df.columns if col in vectorizer.get_feature_names_out()]

df_temp = df[feeling_cols + ['label']].copy()
print(df_temp.groupby('label').mean().T)

label          The Persistence of Memory  The Starry Night  \
anxious                         0.029058          0.008968   
awe                             0.001779          0.053697   
beautiful                       0.002785          0.018729   
beauty                          0.000000          0.027685   
bright                          0.000000          0.014726   
calm                            0.017200          0.174350   
clock                           0.025755          0.000000   
clocks                          0.060357          0.000000   
cold                            0.006778          0.022493   
colors                          0.010588          0.008774   
colours                         0.008731          0.002923   
confused                        0.048932          0.013360   
content                         0.001164          0.016356   
dark                            0.015028          0.014888   
day                             0.008464          0.007503   
excited 

In [ ]:
import re
import numpy as np
import pandas as pd

# Haseeb
# cleaning the worth column

def clean_worth_col(val):
    if pd.isna(val):
        return np.nan

    s = str(val).strip().lower()
    missing_tokens = {
        "", "na", "n/a", "none", "null", "nan",
        "idk", "i dont know", "i don't know",
        "dont know", "don't know", "unknown",
        "priceless", "free"
    }

    if s in missing_tokens:
        return np.nan

    # remove common noise
    s = s.replace(",", "")
    s = s.replace("$", "")
    s = s.replace("cad", "")
    s = s.replace("canadian dollars", "")
    s = s.strip()
    s = s.replace(" ", "")

    # extract the digits so the number only.
    number = re.findall(r'\d+(?:\.\d+)?', s)

    # handle ranges
    if len(number) >= 2 and ("-" in s or "between" in s):
        return (float(number[0]) + float(number[1])) / 2

    # handle words like billion, million ..
    if number:
        num = float(number[0])

        if "billion" in s:
            return num * 1_000_000_000

        if "million" in s:
            return num * 1_000_000

        if re.search(r'\bk\b', s) or s.endswith("k"):
            return num * 1_000

        return num

    return np.nan


def worth_normal(df):
    df["worth_clean"] = df["worth"].apply(clean_worth_col)

    worth_median = df["worth_clean"].median()
    df["worth_clean"] = df["worth_clean"].fillna(worth_median)

    df["worth_log"] = np.log1p(df["worth_clean"])

    mean = df["worth_log"].mean()
    std = df["worth_log"].std()
    df["worth_scaled"] = (df["worth_log"] - mean) / std

    return df

In [ ]:
df = worth_normal(df)
df.drop(columns = {'worth', 'worth_log'}, inplace=True)
df

,rid,label,Class,intensity,sombre?,content?,calm?,uneasy?,num_colors,num_objects,...,strange,summer,things,time,uncomfortable,uneasy,warm,weird,worth_clean,worth_scaled
0,1,The Persistence of Memory,0,7.0,3.0,4.0,4.0,2.0,5.0,3.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,125.0,-0.293527
1,2,The Persistence of Memory,0,5.0,4.0,3.0,2.0,1.0,2.0,4.0,...,0.0,0.0,0.0,0.392321,0.0,0.0,0.0,0.0,0.0,-1.328748
2,3,The Persistence of Memory,0,7.0,4.0,1.0,1.0,4.0,4.0,3.0,...,0.0,0.0,0.0,0.482373,0.0,0.0,0.0,0.0,5.0,-0.945217
3,4,The Persistence of Memory,0,7.0,4.0,1.0,2.0,4.0,10.0,7.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,125.0,-0.293527
4,5,The Persistence of Memory,0,7.0,3.0,4.0,5.0,3.0,4.0,6.0,...,0.0,0.0,0.0,0.387674,0.0,0.0,0.0,0.0,300.0,-0.107124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,558,The Water Lily Pond,2,5.0,2.0,4.0,4.0,2.0,2.0,1.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,35.0,-0.561685
1682,559,The Water Lily Pond,2,9.0,2.0,5.0,5.0,1.0,3.0,3.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,200.0,-0.193560
1683,560,The Water Lily Pond,2,8.0,1.0,5.0,3.0,1.0,2.0,2.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,1.0,0.0,200.0,-0.193560
1684,561,The Water Lily Pond,2,5.0,2.0,2.0,4.0,1.0,4.0,1.0,...,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,10.0,-0.815472


In [ ]:
plt.show()